# Flow Matching for Discrete Data: Sudoku

Two methods applied to *Sudoku Extreme* (9×9 grid, 81-token context), sharing
the same 28M-parameter DiT backbone.

We model categorical sequences $\mathbf{y} = (y^1, \ldots, y^L) \in V^L$ over a
vocabulary of size $|V|$. Each token $k$ has a learned embedding $\mathbf{w}_k$
in some manifold $\mathcal{M}$ — the unit sphere $S^{d-1}$ for the spherical
method, or just token IDs for the masked baseline.

1. **vMF (spherical flow matching).** The forward process puts each token's
   embedding on the sphere and samples noisy states via the von Mises–Fisher
   distribution: at time $t$,
   $\mathbf{x}_t^l \sim \mathrm{vMF}(\mathbf{w}_{y^l}, \kappa(t))$ independently per
   position. The concentration $\kappa(t)$ ramps from $\kappa(0){=}0$ (uniform on
   $S^{d-1}$) to $\kappa(1){=}\kappa_{\max}$. The model is conditioned on the
   noise level via adaLN.
2. **Masked diffusion.** No continuous embedding — discrete CTMC baseline. Each
   token is masked i.i.d. with probability $t$ and the model predicts the
   original vocabulary; CE is computed only on masked positions. Reverse kernel
   unmasks tokens progressively at sampling time.

The notebook is inference-only. Reference: *Spherical Flows for Sampling
Categorical Data*, Chemseddine, Kornhardt, Steidl, 2026
([arXiv:2605.05629](https://arxiv.org/abs/2605.05629)). The model + sampler
code under `flows_categorical/` is copied (with attribution) from the paper's
source repo.


## How training works

Both methods share the same training recipe: **sample a noise level $t$,
corrupt a clean token sequence $\mathbf{y}$ via the forward process, push the
sample through the backbone, and minimize cross-entropy against $\mathbf{y}$.**
For categorical data we factorize over positions; the loss is eq. (17) of the
paper,
$$\mathcal{L}(\theta) \;=\; -\,\mathbb{E}_{t \sim \mathcal{U}(0,1),\;\mathbf{y} \sim p_{\mathrm{data}},\;\mathbf{x}_t \sim p_t(\cdot \mid \mathbf{y})}\!\Bigg[\,\sum_{l=1}^{L} \log p_\theta^l\!\big(y^l \,\big|\, \mathbf{x}_t\big)\,\Bigg],$$
where $p_t(\cdot \mid \mathbf{y})$ is the forward (noising) process and the
per-position predictive distribution is parameterized as a softmax (eq. 16),
$$p_\theta^l(w_k \mid \hat{x}^l) \;=\; \frac{\exp(s_k^l)}{\sum_j \exp(s_j^l)},
\qquad s_k^l \;:=\; \langle \mathbf{w}_k,\, \hat{x}^l \rangle + b_k,$$
with $\hat{\mathbf{x}} = T_\theta(\mathbf{x}_t)$ the backbone output and $b_k \in \mathbb{R}$
learnable per-token biases.

The whole framework rests on **Proposition 4.1** of the paper: the minimizer
of $\mathcal{L}(\theta)$ is the per-position marginal posterior
$p_t^l(\cdot \mid \mathbf{x})$ of the forward process. For vMF paths,
*everything else follows from this posterior*: the marginal velocity (eq. 18)
and the Riemannian score (eq. 19) are both posterior-weighted tangent sums in
the $\mathbf{w}_k$,
$$v_{\theta,t}^l(\mathbf{x}) \;=\; \dot{\kappa}_t \sum_k p_{\theta,t}^l(\mathbf{w}_k|\mathbf{x})\,\tilde\psi_t(\langle \mathbf{w}_k, x^l\rangle)\,P_{x^l}(\mathbf{w}_k),
\qquad
\nabla_{S^{d-1},x^l}\log p_{\theta,t}(\mathbf{x}) \;=\; \kappa_t \sum_k p_{\theta,t}^l(\mathbf{w}_k|\mathbf{x})\,P_{x^l}(\mathbf{w}_k),$$
so the model only ever learns the posterior — one network gives ODE, SDE, and
predictor-corrector sampling for free.

### vMF — forward process lives on the sphere

The vMF density centred on $\mathbf{w} \in S^{d-1}$ with concentration $\kappa \ge 0$ is
$$\varphi(x;\,\mathbf{w},\,\kappa) \;=\; C_d(\kappa)\,\exp\!\big(\kappa\,\langle \mathbf{w},\, x\rangle\big),$$
and the conditional path at position $l$ is $p_t(\cdot \mid w^l) = \varphi(\cdot;\, w^l,\, \kappa_t)$
with $\kappa_t$ a monotone schedule from $\kappa_0 = 0$ (uniform on the sphere)
to $\kappa_{\max} > 0$ (concentrated). Token embeddings $\mathbf{w}_k$ are learned
jointly with $\theta$ and constrained to $S^{d-1}$; the backbone outputs
$\hat{x}^l \in S^{d-1}$ and the softmax above produces per-token logits as cosine
similarities (plus biases) — no multiplicative $\kappa$.

For sampling we discretize the flow ODE in concentration space with Euler steps
$\Delta\kappa_t = \kappa_{t_{n+1}} - \kappa_{t_n}$, using the factorization
$\psi_t = \dot{\kappa}_t\,\tilde\psi_t$ (eq. 13) so that $\dot{\kappa}_t$
cancels and never appears explicitly. The predictor–corrector scheme adds $k$
Langevin steps after each predictor step using the closed-form score above.

The schedule $\kappa_t$ is itself learned (piecewise-linear warp fit to the per-sample CE; Remark 4.2). The backbone receives the normalized noise level $\kappa(t)/\kappa_{\max}$ via adaLN modulation.

### Masked — forward process lives in token space

There are no continuous embeddings. The forward process samples a time
$t \sim \mathcal{U}(0,1)$ and independently replaces each position with a special
`[MASK]` symbol with probability $1-(1-t)^p$ (here $p{=}1$, so the mask rate is
exactly $t$):
$$y_t^l \;=\; \begin{cases} [\mathrm{MASK}] & \text{w.p.\ }t, \\ y^l & \text{w.p.\ }1-t. \end{cases}$$
The backbone applies a standard learned embedding layer internally to convert
$y_t$ into vectors, then outputs logits over $V$ directly. The same
cross-entropy loss is computed only on positions that were masked. At sampling
time a reverse CTMC kernel unmasks tokens progressively (MDLM,
[Sahoo et al., 2024](https://github.com/kuleshov-group/mdlm)).

### Same loss, different state space

Both setups are an expectation over a forward process and a cross-entropy
between predicted and true tokens. They differ only in what gets noised and how
the backbone consumes it:

| | what $\mathbf{x}_t$ is | what enters the backbone |
|---|------|------|
| **vMF** | continuous unit vectors on $S^{d-1}$ | $\mathbf{x}_t$ directly (tied to a softmax head over the $\mathbf{w}_k$) |
| **Masked** | a partly-masked token sequence | $\mathbf{x}_t$ through a learned discrete embedding layer |

Everything else — DiT backbone with adaLN, optimizer schedule, ~1M training
steps at batch 128 — is shared.

> Full training code lives in the source repo
> [`JChemseddine/spherical`](https://github.com/JChemseddine/spherical)
> (branch `paper-release-anon`).


## Setup for Google Colab


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone --depth 1 -q https://github.com/JChemseddine/fm_tutorial.git
    %cd fm_tutorial
    !pip install -q einops huggingface_hub


In [ ]:
import os
import json

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Determinism for the tutorial — same puzzles, same samples for everyone.
SEED = 1234   # also the paper-reproducibility seed for the Sudoku checkpoints
torch.manual_seed(SEED)
np.random.seed(SEED)


## Checkpoints

Two pretrained checkpoints (~440 MB each, 28M params + Adam state + EMA copy)
live at [`Jugc/fm-tutorial`](https://huggingface.co/Jugc/fm-tutorial) on
Hugging Face Hub:

```
Jugc/fm-tutorial/
├── vmf_tc_d11_p1/checkpoint.pt
└── masked_p1/checkpoint.pt
```

The cell below downloads them on first run; subsequent runs use the local HF cache.


In [ ]:
METHOD_NAMES = ["vmf_tc_d11_p1", "masked_p1"]
HF_REPO = "Jugc/fm-tutorial"


def resolve_checkpoint(name):
    # Dev convenience: if you have a local copy in networks/, use that.
    local = os.path.join("networks", name, "final.pt")
    if os.path.exists(local):
        return local
    return hf_hub_download(repo_id=HF_REPO, filename=f"{name}/checkpoint.pt", repo_type="model")


CKPT_PATHS = {name: resolve_checkpoint(name) for name in METHOD_NAMES}
for name, p in CKPT_PATHS.items():
    print(f"  {name:18s}  {p}")


In [ ]:
# Held-out puzzles: bundled in the tutorial repo (100 random puzzles from sudoku-extreme test split)
PUZZLES_PATH = "data/sudoku_extreme_100.npz"
data = np.load(PUZZLES_PATH)
test_inputs = torch.from_numpy(data["inputs"].astype(np.int64))   # (100, 81), 0=blank, 1-9=clue
test_labels = torch.from_numpy(data["labels"].astype(np.int64))   # (100, 81), 1-9 (full solution)

assert test_inputs.min() >= 0 and test_inputs.max() <= 9
assert test_labels.min() >= 1 and test_labels.max() <= 9

print(f"Loaded {len(test_inputs)} test puzzles, shape {tuple(test_inputs.shape)}")
print(f"Avg clues per puzzle: {(test_inputs > 0).float().sum(dim=1).mean():.1f}  "
      f"(min {(test_inputs > 0).sum(dim=1).min().item()}, max {(test_inputs > 0).sum(dim=1).max().item()})")


## 2. Anatomy of a Sudoku puzzle

Each puzzle is a flat sequence of length 81 (the 9×9 grid in row-major order).
Blanks are token 0; the model must predict tokens 1–9 for those positions.
*Sudoku Extreme* puzzles have as few as 17 clues — they are deliberately hard.


In [ ]:
def show_grid(tokens, clue_mask=None, gt=None, title=None, ax=None):
    '''Plot a (81,) tensor as a 9x9 grid.
    - clue cells: bold black
    - non-clue cells: blue if they match `gt` (when provided), red if wrong
    - if `gt` is None, all non-clue cells are blue (no correctness signal)
    '''
    if ax is None:
        _, ax = plt.subplots(figsize=(3.2, 3.2))
    grid = tokens.view(9, 9).cpu().numpy()
    ax.set_xlim(0, 9); ax.set_ylim(9, 0)
    ax.set_xticks([]); ax.set_yticks([])
    for x in range(10):
        lw = 2.0 if x % 3 == 0 else 0.5
        ax.plot([x, x], [0, 9], "k-", lw=lw)
        ax.plot([0, 9], [x, x], "k-", lw=lw)
    cm = None if clue_mask is None else clue_mask.view(-1).cpu().numpy()
    gt_flat = None if gt is None else gt.view(-1).cpu().numpy()
    tok_flat = tokens.view(-1).cpu().numpy()
    for i in range(9):
        for j in range(9):
            idx = i * 9 + j
            v = int(grid[i, j])
            if v == 0:
                continue
            is_clue = cm is not None and bool(cm[idx])
            if is_clue:
                color, weight = "black", "bold"
            elif gt_flat is not None and int(tok_flat[idx]) != int(gt_flat[idx]):
                color, weight = "#d62728", "normal"   # red for wrong
            else:
                color, weight = "#1f77b4", "normal"   # blue for correct (or no gt)
            ax.text(j + 0.5, i + 0.5, str(v), ha="center", va="center",
                    fontsize=14, fontweight=weight, color=color)
    if title:
        ax.set_title(title, fontsize=10)
    return ax


idx = 0
clue_mask = (test_inputs[idx] != 0)
fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.2))
show_grid(test_inputs[idx], clue_mask, title="Puzzle (clues bold)", ax=axes[0])
show_grid(test_labels[idx], clue_mask, title="Ground-truth solution", ax=axes[1])
plt.tight_layout(); plt.show()


## 3. Loading the two methods

Each checkpoint ships with the training `config.json` and a `checkpoint.pt`
holding the model weights and (for vMF) the learned $\kappa(t)$ schedule parameters.
A small helper hides the boilerplate.


In [ ]:
from flows_categorical.config import Config
from flows_categorical.model.backbone import ContinuousTransformer, MaskedTransformer
from flows_categorical.methods import create_sampler
from flows_categorical.methods.continuous.spherical.vmf import PsiTable
from flows_categorical.schedule.cdcd_warp import CDCDWarp


def load_method(name, device=DEVICE, use_ema=True):
    """Load (config, model, sampler) for a method name. Uses EMA weights by default."""
    ckpt_path = CKPT_PATHS[name]
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    config = Config.from_dict(ckpt["config"])

    # Backbone
    if config.flow.noise_process == "masked":
        model = MaskedTransformer(config.model.vocab_size, config.model)
    else:
        model = ContinuousTransformer(config.model.vocab_size, config.model)

    # Pick EMA if asked & available; else raw model_state_dict
    src_sd = ckpt.get("ema_state_dict") if use_ema else None
    if src_sd is None:
        src_sd = ckpt["model_state_dict"]
        weight_src = "model_state_dict"
    else:
        weight_src = "ema_state_dict"
    msd = {k.removeprefix("_orig_mod."): v for k, v in src_sd.items()}
    missing, unexpected = model.load_state_dict(msd, strict=False)
    if missing:
        print(f"  [{name}] missing keys: {missing[:3]}{'...' if len(missing) > 3 else ''}")
    if unexpected:
        print(f"  [{name}] unexpected keys: {unexpected[:3]}{'...' if len(unexpected) > 3 else ''}")
    model.to(device).eval()

    # Sampler-side extras
    sampler_kwargs = {}
    if config.flow.noise_process == "vmf":
        sampler_kwargs["psi_table"] = PsiTable(
            config.model.embed_dim,
            kappa_range=config.flow.kappa_max,
            grid_size=config.flow.psi_grid_size,
        )
    if config.flow.use_warp and "warp_state" in ckpt:
        warp = CDCDWarp(
            kappa_max=config.flow.kappa_max,
            num_bins=config.flow.warp_bins,
            warmup_steps=config.flow.time_warp_warmup,
            ema_decay=config.flow.warp_ema_decay,
            noise_increasing=False,  # vMF: parameter (kappa) increases with signal
        )
        warp.load_state_dict(ckpt["warp_state"])
        warp.to(device).eval()
        sampler_kwargs["warp"] = warp

    sampler = create_sampler(model, config, **sampler_kwargs)
    print(f"  [{name}] loaded from {weight_src}  (step {ckpt.get('step','?')})")
    return config, model, sampler


methods = {}
for name in METHOD_NAMES:
    print(f"Loading {name}...")
    methods[name] = load_method(name)

# Quick model-size summary
print()
for name, (cfg, mdl, _) in methods.items():
    n_params = sum(p.numel() for p in mdl.parameters())
    print(f"  {name:18s}  {n_params/1e6:5.1f}M params  noise={cfg.flow.noise_process}")


## 4. Solving one puzzle with each method

Same puzzle, same clues. The model conditions on the clue positions via
`clue_mask` (which positions are observed) and `clue_values` (the digits at
those positions). The sampler pins the clue embeddings throughout the trajectory.


In [ ]:
torch.manual_seed(SEED)

def sample_one(sampler, puzzle, num_samples=1, return_intermediates=False):
    """Sample completions for a single puzzle. Returns (tokens, intermediates)."""
    puzzle = puzzle.to(DEVICE)
    clue_mask = (puzzle != 0).unsqueeze(0).expand(num_samples, -1).contiguous()
    clue_values = puzzle.unsqueeze(0).expand(num_samples, -1).contiguous()
    out = sampler.sample(
        num_samples=num_samples, device=DEVICE,
        clue_mask=clue_mask, clue_values=clue_values,
        return_intermediates=return_intermediates, verbose=False,
    )
    return out  # dict with 'tokens', possibly 'intermediates'


idx = 0
puzzle = test_inputs[idx]
clue_mask = (puzzle != 0)
gt = test_labels[idx]

results = {}
for name, (_, _, sampler) in methods.items():
    out = sample_one(sampler, puzzle, num_samples=1)
    results[name] = out["tokens"][0].cpu()

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
show_grid(puzzle, clue_mask, title="Puzzle", ax=axes[0])
for ax, (name, tokens) in zip(axes[1:], results.items()):
    correct = (tokens == gt).float().mean().item() * 100
    show_grid(tokens, clue_mask, gt=gt, title=f"{name}\n{correct:.0f}% cell-correct", ax=ax)
plt.tight_layout(); plt.show()


## Sampler knobs

For the spherical (vMF) sampler we use a predictor–corrector ODE on $S^{d-1}$.
Each predictor step is one Euler step along the vMF-derived velocity; an
optional Langevin corrector then takes `CORRECTOR_STEPS` score-based moves
on the tangent plane, applied every `CORRECTOR_INTERVAL` predictor steps.

$$\mathrm{NFE} \;=\; n_{\mathrm{pred}}\;+\;\Big\lfloor \tfrac{n_{\mathrm{pred}}}{n_{\mathrm{int}}} \Big\rfloor \cdot k_{\mathrm{corrector}}.$$

Paper sweeps at total NFE=128 (with interval $n_{\mathrm{int}}=1$):
$(n_{\mathrm{pred}}, k_{\mathrm{corrector}}) \in \{(64,1), (32,3), (16,7)\}$.

| Knob | What it does |
|------|--------------|
| `PREDICTOR_STEPS` ($n_{\mathrm{pred}}$) | Number of Euler ODE steps along the velocity field. |
| `CORRECTOR_STEPS` ($k_{\mathrm{corrector}}$) | Langevin corrector steps per correction (vMF only). `0` = plain ODE. |
| `CORRECTOR_INTERVAL` ($n_{\mathrm{int}}$) | Apply the corrector every $n_{\mathrm{int}}$-th predictor step. `1` = every step. |
| `CORRECTOR_EPS` | Langevin step size $\varepsilon$. |

The masked sampler has no corrector — only `PREDICTOR_STEPS` (= total NFE for that method) is in scope. The mask-rate schedule $t^p$ is set at training time and is not tunable at inference.


In [ ]:
torch.manual_seed(SEED)

# --- Edit me ---
PREDICTOR_STEPS    = 32   # ODE Euler steps along the velocity field
CORRECTOR_STEPS    = 1    # Langevin corrector steps per correction (vMF only; 0 = plain ODE)
CORRECTOR_INTERVAL = 1    # apply corrector every N-th predictor step (1 = every step)
CORRECTOR_EPS      = 0.01 # Langevin step size eps
# -----------------

n_corrections = PREDICTOR_STEPS // CORRECTOR_INTERVAL
TOTAL_NFE = PREDICTOR_STEPS + n_corrections * CORRECTOR_STEPS  # NFE budget shared across both methods
print(f"vMF: {PREDICTOR_STEPS} predictor + {n_corrections}*{CORRECTOR_STEPS} corrector = {TOTAL_NFE} NFE")
print(f"masked: {TOTAL_NFE} predictor steps  (NFE-matched to vMF)")

_, _, sampler = methods["vmf_tc_d11_p1"]
sampler.num_steps          = PREDICTOR_STEPS
sampler.corrector_steps    = CORRECTOR_STEPS
sampler.corrector_interval = CORRECTOR_INTERVAL
sampler.corrector_epsilon  = CORRECTOR_EPS
sampler.method = "pc_softmax"  # ODE predictor + Langevin corrector (becomes plain ODE when CORRECTOR_STEPS=0)

methods["masked_p1"][2].num_steps = TOTAL_NFE


## Predictor–corrector vs plain ODE at the same NFE budget

The Langevin corrector spends extra forward passes refining each predictor
step. At equal total NFE, is it worth it? Two settings with the same total
budget $N$:

- **Plain ODE** — all NFE goes into predictor steps: $(n_{\mathrm{pred}}, k) = (N, 0)$.
- **PC** — half predictor, one corrector per predictor: $(n_{\mathrm{pred}}, k) = (N/2,\, 1)$, NFE $= N/2 + N/2 = N$.

Same puzzle. The masked sampler has no corrector, so it is shown at the same
NFE budget but as a single sample for reference.


In [ ]:
torch.manual_seed(SEED)

# Use the knobs from above as the PC setting; derive ODE-only at the same total NFE.
pc_pred, pc_corr, pc_int = PREDICTOR_STEPS, CORRECTOR_STEPS, CORRECTOR_INTERVAL
pc_nfe   = pc_pred + (pc_pred // pc_int) * pc_corr
ode_pred = pc_nfe
print(f"PC : (n_pred={pc_pred}, k={pc_corr}, interval={pc_int})  -> NFE={pc_nfe}")
print(f"ODE: (n_pred={ode_pred}, k=0)                            -> NFE={ode_pred}")
print(f"masked: {pc_nfe} predictor steps  (NFE-matched, no corrector available)")


def run_vmf(sampler, predictor_steps, corrector_steps, corrector_interval, corrector_eps=CORRECTOR_EPS):
    sampler.num_steps          = predictor_steps
    sampler.corrector_steps    = corrector_steps
    sampler.corrector_interval = corrector_interval
    sampler.corrector_epsilon  = corrector_eps
    sampler.method = "pc_softmax"
    return sample_one(sampler, puzzle)["tokens"][0].cpu()


def run_masked(sampler, num_steps):
    sampler.num_steps = num_steps
    return sample_one(sampler, puzzle)["tokens"][0].cpu()


fig, axes = plt.subplots(2, 3, figsize=(10, 6.4))

show_grid(puzzle, clue_mask, title="vmf_tc_d11_p1", ax=axes[0, 0])
_, _, sampler = methods["vmf_tc_d11_p1"]
for col, (label, pred, corr, interval) in enumerate(
    [("ODE", ode_pred, 0, 1), ("PC", pc_pred, pc_corr, pc_int)], start=1
):
    tokens = run_vmf(sampler, pred, corr, interval)
    acc = (tokens == gt).float().mean().item() * 100
    nfe = pred + (pred // interval) * corr
    show_grid(tokens, clue_mask, gt=gt,
              title=f"{label}  (n_pred={pred}, k={corr})\nNFE={nfe}  acc={acc:.0f}%",
              ax=axes[0, col])

show_grid(puzzle, clue_mask, title="masked_p1", ax=axes[1, 0])
masked_tokens = run_masked(methods["masked_p1"][2], pc_nfe)
masked_acc = (masked_tokens == gt).float().mean().item() * 100
show_grid(masked_tokens, clue_mask, gt=gt,
          title=f"masked (no corrector)\nNFE={pc_nfe}  acc={masked_acc:.0f}%",
          ax=axes[1, 1])
axes[1, 2].axis("off")
axes[1, 2].text(0.5, 0.5, "— no corrector\n   for masked diffusion —",
                ha="center", va="center", fontsize=11, color="gray", transform=axes[1, 2].transAxes)

plt.tight_layout(); plt.show()


## Visualizing the trajectory

We capture one full sample with each method (`return_intermediates=True` so the
sampler stores logits at every step) and then look at how the trajectory
evolves over time.


In [ ]:
torch.manual_seed(SEED)

# Trace both methods with intermediates so we can compare state vs posterior side-by-side.
cfg_v, mdl_v, sampler_v = methods["vmf_tc_d11_p1"]
cfg_m, mdl_m, sampler_m = methods["masked_p1"]

sampler_v.num_steps          = 32
sampler_v.method             = "pc_softmax"
sampler_v.corrector_steps    = 1
sampler_v.corrector_interval = 1
out_v = sample_one(sampler_v, puzzle, num_samples=1, return_intermediates=True)
trace = out_v["intermediates"]

sampler_m.num_steps = 32  # match the vmf trace length
out_m = sample_one(sampler_m, puzzle, num_samples=1, return_intermediates=True)
trace_m = out_m["intermediates"]

# Backfill 'logits' per snapshot if the sampler version on disk doesn't already
# save them (Colab can cache an older flows_categorical/ across runs).
def backfill_logits(trace_, model, is_continuous, cfg=None):
    if "logits" in trace_[0]:
        return
    with torch.no_grad():
        for snap in trace_:
            if is_continuous:
                h = snap["h_t"].to(DEVICE)
                sigma = (torch.full((h.shape[0],), snap["kappa"] / cfg.flow.kappa_max, device=DEVICE)
                         if cfg.model.time_conditioning else None)
                h_prime = model(h, clue_mask=None, sigma=sigma)
                snap["logits"] = model.compute_logits(h_prime, W_E=model.get_W_E()).detach()
            else:
                snap["logits"] = model(snap["x_t"].to(DEVICE), clue_mask=None).detach()

backfill_logits(trace,   mdl_v, is_continuous=True,  cfg=cfg_v)
backfill_logits(trace_m, mdl_m, is_continuous=False)

print(f"vMF    trace: {len(trace)} snapshots  (kappa {trace[0]['kappa']:.2f} -> {trace[-1]['kappa']:.2f})")
print(f"masked trace: {len(trace_m)} snapshots  (mask rate {trace_m[0]['mask_rate']:.2f} -> {trace_m[-1]['mask_rate']:.2f})")


### State vs posterior, vMF vs masked

Two complementary views of each trajectory:

- **State row.** The "current decoding" of $\mathbf{x}_t$ itself.
  - For **vMF**: argmax of $s_k^l = \langle \mathbf{w}_k, x_t^l \rangle + b_k$ — same
    cosine-logits formula as eq. (16) of the paper, but applied to the noisy
    state $x_t$ instead of the backbone output $\hat{x}$. The state is a
    continuous direction on the sphere, so this decoding is *soft* and can
    swap between digits at every step until the trajectory ends.
  - For **masked**: the current token at each position. Once a position is
    unmasked, it is **locked in** — masked diffusion cannot revisit that
    decision.
- **Posterior row.** The model's clean-token estimate at the current state,
  argmax of $p_\theta(\cdot \mid \mathbf{x}_t)$. (This is what the bottom-left
  panel of the side-by-side display higher up converges to.)

Watch the masked state row fill in monotonically while the vMF state row keeps
shifting until $\kappa$ is large enough to commit. Red = the decoded digit
disagrees with the ground-truth solution; blue = matches; clue cells stay bold
black.


In [ ]:
def viz_state_vs_posterior(trace_v, mdl_v, trace_m, mdl_m, puzzle, gt, n_panels=8):
    '''4-row figure: vMF state, vMF posterior, masked state, masked posterior.

    The vMF "state" is decoded directly from the sphere via
    s_k = <w_k, x_t> + b_k (same logits formula as eq. 16, but applied to the
    raw noisy state x_t instead of the backbone output x_hat).  This is a soft,
    re-decodable view of where the trajectory currently sits.

    The masked "state" is just the token x_t: digits or [MASK].  Once unmasked
    a position is locked.
    '''
    clue_mask_flat = (puzzle != 0).view(-1).cpu()
    clue_vals = puzzle.view(-1).cpu()
    gt_flat = gt.view(-1).cpu()

    idxs_v = np.linspace(0, len(trace_v) - 1, n_panels).astype(int)
    idxs_m = np.linspace(0, len(trace_m) - 1, n_panels).astype(int)

    fig, axes = plt.subplots(4, n_panels, figsize=(2.4 * n_panels, 10.4))

    def grid_lines(ax):
        ax.set_xlim(0, 9); ax.set_ylim(9, 0); ax.set_xticks([]); ax.set_yticks([])
        for x in range(10):
            lw = 1.5 if x % 3 == 0 else 0.3
            ax.plot([x, x], [0, 9], "k-", lw=lw)
            ax.plot([0, 9], [x, x], "k-", lw=lw)

    def draw_argmax(ax, argmax_1_to_9, confidence):
        grid_lines(ax)
        for i in range(9):
            for j in range(9):
                idx = i * 9 + j
                if bool(clue_mask_flat[idx]):
                    v = int(clue_vals[idx])
                    color, weight, alpha = "black", "bold", 1.0
                else:
                    v = int(argmax_1_to_9[idx])
                    color = "#d62728" if v != int(gt_flat[idx]) else "#1f77b4"
                    weight = "normal"
                    raw = float(confidence[idx])
                    alpha = 0.25 + 0.75 * max(0.0, (raw - 1/9) / (1.0 - 1/9))
                ax.text(j + 0.5, i + 0.5, str(v), ha="center", va="center",
                        fontsize=10, alpha=alpha, color=color, fontweight=weight)

    def draw_masked_state(ax, x_t_int, mask_id):
        grid_lines(ax)
        for i in range(9):
            for j in range(9):
                idx = i * 9 + j
                v_raw = int(x_t_int[idx])
                if v_raw == mask_id:
                    continue  # blank cell for unrevealed mask
                if bool(clue_mask_flat[idx]):
                    color, weight = "black", "bold"
                else:
                    color = "#d62728" if v_raw != int(gt_flat[idx]) else "#1f77b4"
                    weight = "normal"
                ax.text(j + 0.5, i + 0.5, str(v_raw), ha="center", va="center",
                        fontsize=10, color=color, fontweight=weight)

    # --- Row 0/1: vMF state (cos-sim of x_t) / vMF posterior ---
    W_E_v = mdl_v.get_W_E().detach()  # (d, V)
    for col, k in enumerate(idxs_v):
        snap = trace_v[k]
        x_t = snap["h_t"].to(DEVICE)  # (1, L, d)
        with torch.no_grad():
            state_logits = mdl_v.compute_logits(x_t, W_E=W_E_v)
        state_d = F.softmax(state_logits[0, :, 1:10], dim=-1).cpu()
        state_argmax = state_d.argmax(dim=-1) + 1
        state_conf = state_d.max(dim=-1).values

        post_d = F.softmax(snap["logits"][0, :, 1:10], dim=-1).cpu()
        post_argmax = post_d.argmax(dim=-1) + 1
        post_conf = post_d.max(dim=-1).values

        draw_argmax(axes[0, col], state_argmax, state_conf)
        draw_argmax(axes[1, col], post_argmax, post_conf)
        axes[0, col].set_title(f"$\\kappa$={snap['kappa']:.1f}", fontsize=9)

    # --- Row 2/3: masked state / masked posterior ---
    mask_id = mdl_m.mask_token_id
    for col, k in enumerate(idxs_m):
        snap = trace_m[k]
        x_t_tokens = snap["x_t"][0].cpu()

        post_d = F.softmax(snap["logits"][0, :, 1:10], dim=-1).cpu()
        post_argmax = post_d.argmax(dim=-1) + 1
        post_conf = post_d.max(dim=-1).values

        draw_masked_state(axes[2, col], x_t_tokens, mask_id)
        draw_argmax(axes[3, col], post_argmax, post_conf)
        axes[2, col].set_title(f"mask rate={snap['mask_rate']:.2f}", fontsize=9)

    row_labels = [
        "vMF\nstate $x_t$\n(cos-sim decode)",
        "vMF\nmodel posterior\n($\\arg\\max p_\\theta$)",
        "masked\nstate $x_t$\n(locked tokens)",
        "masked\nmodel posterior\n($\\arg\\max p_\\theta$)",
    ]
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=9, rotation=0,
                                ha="right", va="center", labelpad=8)
    plt.tight_layout(); plt.show()


viz_state_vs_posterior(trace, mdl_v, trace_m, mdl_m, puzzle, gt)


### Viz 2 — per-cell entropy heatmap

At each snapshot we compute the entropy of the per-cell predictive distribution
over the 9 digits, $H(p_i) = -\sum_v p_i(v) \log p_i(v)$. High entropy = the model
is uncertain about cell $i$; near 0 = the model has committed. Clue cells stay at 0.


In [ ]:
def viz_entropy_strip(trace, puzzle, n_panels=8):
    '''Per-cell predictive entropy at each snapshot, normalized to [0, 1].
    Bright = uncertain (entropy near log 9), dark = committed.  Clue cells stay at 0.
    '''
    clue_mask_grid = (puzzle != 0).view(9, 9).numpy()
    idxs = np.linspace(0, len(trace) - 1, n_panels).astype(int)
    fig, axes = plt.subplots(1, n_panels, figsize=(2.4 * n_panels, 2.7))
    log9 = np.log(9.0)
    last_im = None
    for ax, k in zip(axes, idxs):
        snap = trace[k]
        p = F.softmax(snap["logits"][0, :, 1:10], dim=-1)
        H = -(p * p.clamp_min(1e-12).log()).sum(dim=-1).cpu().numpy() / log9
        H = H.reshape(9, 9)
        H[clue_mask_grid] = 0.0
        last_im = ax.imshow(H, cmap="viridis", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        for x in range(10):
            lw = 1.5 if x % 3 == 0 else 0.3
            ax.plot([x - 0.5, x - 0.5], [-0.5, 8.5], "w-", lw=lw)
            ax.plot([-0.5, 8.5], [x - 0.5, x - 0.5], "w-", lw=lw)
        t = snap.get("time", k / max(1, len(trace) - 1))
        kappa = snap.get("kappa", float("nan"))
        ax.set_title(f"t={t:.2f},  $\\kappa$={kappa:.1f}", fontsize=9)
    fig.colorbar(last_im, ax=axes.ravel().tolist(), shrink=0.7, label="entropy / $\\log 9$")
    plt.show()


viz_entropy_strip(trace, puzzle)


## 7. Solving a batch + validity check

For each of `N_PUZZLES` test puzzles we generate one completion with each
method, then check three things:

1. **Cell accuracy** — fraction of non-clue cells matching the ground-truth solution.
2. **Strict-match** — all 81 cells correct.
3. **Sudoku validity** — every row, column, and 3×3 box contains digits 1–9 exactly once.

Validity is a stricter signal than cell accuracy: a method can get 95% of cells
right and still produce an invalid grid.


In [ ]:
torch.manual_seed(SEED)

def is_valid_sudoku(grid_81):
    g = grid_81.view(9, 9)
    target = torch.arange(1, 10)
    for i in range(9):
        if not torch.equal(torch.sort(g[i]).values, target): return False
        if not torch.equal(torch.sort(g[:, i]).values, target): return False
    for bi in range(3):
        for bj in range(3):
            box = g[3*bi:3*bi+3, 3*bj:3*bj+3].flatten()
            if not torch.equal(torch.sort(box).values, target): return False
    return True


N_PUZZLES = 8  # bump up for a sturdier estimate; ~seconds per puzzle on T4

# Three configurations, all at the same total NFE = TOTAL_NFE (set in the knob cell above).
# - vmf_tc-ODE: plain ODE predictor, no corrector
# - vmf_tc-PC : ODE + Langevin corrector
# - masked   : MDLM-style CTMC sampler (no corrector concept)
ode_pred = TOTAL_NFE
pc_pred  = PREDICTOR_STEPS
pc_corr  = CORRECTOR_STEPS
pc_int   = CORRECTOR_INTERVAL

_, _, sampler_v = methods["vmf_tc_d11_p1"]
sampler_m       = methods["masked_p1"][2]

def set_vmf(num_steps, corrector_steps, corrector_interval):
    sampler_v.num_steps          = num_steps
    sampler_v.corrector_steps    = corrector_steps
    sampler_v.corrector_interval = corrector_interval
    sampler_v.corrector_epsilon  = CORRECTOR_EPS
    sampler_v.method = "pc_softmax"

configs = [
    ("vmf_tc-ODE", lambda: (set_vmf(ode_pred, 0, 1),                  sampler_v)[1]),
    ("vmf_tc-PC",  lambda: (set_vmf(pc_pred, pc_corr, pc_int),        sampler_v)[1]),
    ("masked",     lambda: (setattr(sampler_m, "num_steps", TOTAL_NFE), sampler_m)[1]),
]
stats = {label: dict(cell_acc=[], strict=[], valid=[]) for label, _ in configs}

print(f"All configs at NFE={TOTAL_NFE}.")
print(f"Sampling {N_PUZZLES} puzzles x {len(configs)} configs...")

for idx in range(N_PUZZLES):
    p   = test_inputs[idx]
    gtp = test_labels[idx]
    non_clue = (p == 0)
    for label, prepare in configs:
        sampler = prepare()
        tokens = sample_one(sampler, p, num_samples=1)["tokens"][0].cpu()
        stats[label]["cell_acc"].append((tokens[non_clue] == gtp[non_clue]).float().mean().item())
        stats[label]["strict"].append(bool((tokens == gtp).all()))
        stats[label]["valid"].append(is_valid_sudoku(tokens))

print(f"\nResults over {N_PUZZLES} held-out Sudoku-Extreme puzzles  (NFE={TOTAL_NFE}):")
print(f"  {'config':<14s}  cell-acc   strict   valid")
for label, d in stats.items():
    print(f"  {label:<14s}  {np.mean(d['cell_acc'])*100:6.1f}%   "
          f"{np.mean(d['strict'])*100:5.1f}%   {np.mean(d['valid'])*100:5.1f}%")


## Summary

vMF and masked diffusion give two routes to the same generative target,
sharing the same backbone, training recipe (sample $\mathbf{x}_t$, push it through
the model, cross-entropy against $\mathbf{x}_0$) and decoding step (argmax of
$p_\theta(\cdot \mid \mathbf{x}_t)$). The difference is just whether the noisy state
$\mathbf{x}_t$ lives on $S^{d-1}$ or in token space — soft, re-decodable
direction on the sphere vs. discrete tokens that get locked in once unmasked.

---

**Paper:** *Spherical Flows for Sampling Categorical Data*, Chemseddine,
Kornhardt, Steidl, 2026 —
[arXiv:2605.05629](https://arxiv.org/abs/2605.05629).

The masked-diffusion sampler is based on
[MDLM (Sahoo et al., 2024)](https://github.com/kuleshov-group/mdlm); the DiT
backbone follows Peebles & Xie, 2023.
